# Task 2, Part C1: Validating the Full-Precision KV-Cache Reference

Model `Qwen/Qwen2.5-1.5B-Instruct`, FP16, SDPA, greedy decoding, batch 1, the staff retrieval JSONL (180 prompts =
2 lengths x 3 evidence positions x 3 distractor counts x 10 cases). Prompts use the model's chat template; the
context/suffix split is the character position of `QUESTION_BEGIN`; evidence character spans are mapped to tokens
(used for evaluation only, never given to a policy).

Checks before any compressed variant:
1. direct full-forward logits == split prefill + suffix logits;
2. a cache rebuilt through the eviction code path with 100% retention reproduces ordinary generation (HF `generate`);
3. retained tokens keep their absolute RoPE positions (using physical positions instead changes logits);
4. all logits finite; full-cache retrieval accuracy for every prompt cell.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import torch
from shared.src.io import read_jsonl, save_json, write_jsonl
from shared.src.env import environment_record
from task2_attention_kv.src import kv_cache as KV

model, tok = KV.load_kv_model()
cases_raw = read_jsonl(ctx.data / "kv_retrieval" / "kv_retrieval_cases.jsonl.gz")
cases = [KV.prepare_case(tok, c) for c in cases_raw]
OUT = ctx.results / "kv"
OUT.mkdir(exist_ok=True)
save_json(OUT / "environment.json", environment_record(dict(model=KV.MODEL_NAME, revision=model._pa4_revision, dtype="float16",
                                                          attn=model.config._attn_implementation, dims=KV.kv_dims(model))))
lens = pd.DataFrame([dict(id=c["id"], nominal=c["nominal_len"], position=c["position"], distractors=c["distractors"],
                          total_tokens=c["n_tokens"], ctx_tokens=c["ctx_len"], suffix_tokens=c["suffix_len"],
                          evidence_tokens=len(c["evidence_tokens"]), evidence_first=c["evidence_tokens"][0]) for c in cases])
lens.to_csv(ctx.splits / "kv_prompt_token_lengths.csv", index=False)
lens.groupby(["nominal", "position"])[["total_tokens", "ctx_tokens", "suffix_tokens", "evidence_first"]].agg(["min", "max"])

## 1-3. Split prefill, 100% retention, position preservation

In [ ]:
checks = []
for c in [cases[0], cases[95], cases[179]]:
    ids, Lc = c["ids"], c["ctx_len"]
    with torch.no_grad():
        full = model(input_ids=torch.tensor([ids], device="cuda")).logits[0].float()
        cache, _ = KV.prefill(model, ids[:Lc])
        lg, cache = KV.forward_step(model, cache, ids[Lc:], Lc)
        gen = model.generate(torch.tensor([ids], device="cuda"), max_new_tokens=16, do_sample=False,
                             pad_token_id=tok.eos_token_id)[0, len(ids):].tolist()
    keep = KV.run_policy(model, tok, c, KV.KeepAll())
    fp = KV.run_policy(model, tok, c, KV.Policy())
    gen_text = tok.decode(gen, skip_special_tokens=True)
    # position preservation: evict the first 100 tokens; absolute vs physical RoPE positions for the suffix
    with torch.no_grad():
        cc, _ = KV.prefill(model, ids[:Lc]); e = KV._evict(cc, list(range(100, Lc)))
        lg_abs, _ = KV.forward_step(model, e, ids[Lc:], Lc)
        cc, _ = KV.prefill(model, ids[:Lc]); e = KV._evict(cc, list(range(100, Lc)))
        lg_phys, _ = KV.forward_step(model, e, ids[Lc:], Lc - 100)
    checks.append(dict(id=c["id"], split_vs_full_max_abs=float((lg - full[Lc:]).abs().max()),
                       split_vs_full_argmax_agree=float((lg.argmax(-1) == full[Lc:].argmax(-1)).float().mean()),
                       all_finite=bool(torch.isfinite(full).all() and torch.isfinite(lg).all()),
                       generate_text=gen_text, keepall_text=keep["text"], fp16_text=fp["text"],
                       keepall_equals_generate=keep["text"] == gen_text, keepall_kl_vs_fp16=None,
                       abs_vs_phys_positions_max_abs=float((lg_abs - lg_phys).abs().max())))
save_json(OUT / "C1_validation.json", checks)
pd.DataFrame(checks)

## 4. Full FP16 cache retrieval accuracy on every prompt cell

In [ ]:
rows = []
for c in cases:
    r = KV.run_policy(model, tok, c, KV.Policy())
    r.pop("_tf_logits"); r.pop("kept_fp16", None)
    rows.append(r)
write_jsonl(OUT / "C1_fp16_reference.jsonl", rows)
fp = pd.DataFrame(rows)
print("overall exact match:", fp.correct.mean(), " finite NLL:", np.isfinite(fp.answer_nll).all())
fp.pivot_table(index=["nominal_len", "position"], columns="distractors", values="correct", aggfunc="mean")

In [ ]:
fp.groupby("nominal_len")[["correct", "answer_nll", "prefill_ms", "decode_ms", "peak_allocated_mb"]].mean()